# Agrupación: K-means, DBSCAN y Clustering Jerárquico

**Universidad de los Andes — Minería de Datos**  
**Semana 9 — Agrupación Avanzada**

---

En esta sesión profundizamos en los tres algoritmos de clustering más utilizados en la práctica. La Semana 8 introdujo los conceptos fundamentales (qué es clustering, métricas de similitud, métricas de calidad); aquí nos enfocamos en el funcionamiento interno de cada algoritmo, sus fortalezas, sus limitaciones y cuándo elegir cada uno.

**Contenido:**
1. Recapitulación de conceptos clave
2. K-means en profundidad
3. DBSCAN
4. Clustering Jerárquico (Aglomerativo)
5. Comparación de métodos
6. Caso práctico con datos reales

## 0. Configuración del entorno

In [ ]:
# ── Importaciones generales ───────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

# ── Configuración de gráficas ─────────────────────────────────────────────────
%matplotlib inline
mpl.rc('axes', labelsize=13)
mpl.rc('xtick', labelsize=11)
mpl.rc('ytick', labelsize=11)
mpl.rc('figure', figsize=(10, 6))

# ── Semilla para reproducibilidad ────────────────────────────────────────────
np.random.seed(42)

print('Entorno listo. NumPy:', np.__version__, '| pandas:', pd.__version__)

In [ ]:
# ── Importaciones de sklearn ──────────────────────────────────────────────────
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.neighbors import NearestNeighbors

# ── Importaciones de scipy para dendrogramas ──────────────────────────────────
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

print('Todas las librerías importadas correctamente.')

---
## 1. Recapitulación — Conceptos clave de la Semana 8

Antes de profundizar en los algoritmos, recordemos los conceptos fundamentales:

### ¿Qué es el clustering?
El **clustering** (o agrupación) es una tarea de **aprendizaje no supervisado** que busca organizar instancias en grupos (**clusters**) de tal manera que:
- Los puntos dentro del mismo cluster sean **similares entre sí** (cohesión interna alta).
- Los puntos en diferentes clusters sean **disímiles** (separación entre clusters alta).

### Métricas de similitud / distancia
| Métrica | Fórmula | Cuándo usarla |
|---------|---------|---------------|
| Euclidiana | $d = \sqrt{\sum_i (x_i - y_i)^2}$ | Datos continuos, escala similar |
| Manhattan | $d = \sum_i |x_i - y_i|$ | Datos con outliers |
| Coseno | $d = 1 - \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\| \|\mathbf{y}\|}$ | Texto, vectores de alta dimensión |

### Métricas de calidad
- **Inercia (WCSS):** suma de distancias cuadradas de cada punto a su centroide. Menor es mejor, pero siempre disminuye al aumentar $k$.
- **Coeficiente de Silhouette:** $s = \frac{b - a}{\max(a, b)}$ donde $a$ = distancia media intracluster, $b$ = distancia media al cluster vecino más cercano. Rango: $[-1, 1]$; valores cercanos a 1 indican clusters bien definidos.
- **Índice de Davies-Bouldin:** razón entre dispersión interna y separación entre clusters. Menor es mejor.

> **Punto clave:** Ninguna métrica es perfecta en aislamiento. Siempre use al menos dos métricas complementarias y valide visualmente.

---
## 2. K-means en Profundidad

### 2.1 Algoritmo paso a paso

K-means es el algoritmo de clustering más popular por su simplicidad y eficiencia computacional. El algoritmo sigue los siguientes pasos:

**Entrada:** número de clusters $k$, conjunto de datos $\mathbf{X}$.

1. **Inicialización:** seleccionar $k$ centroides iniciales $\mu_1, \mu_2, \ldots, \mu_k$ (aleatoriamente o con K-means++).
2. **Asignación:** asignar cada punto $x_i$ al centroide más cercano:
   $$c_i = \arg\min_j \| x_i - \mu_j \|^2$$
3. **Actualización:** recalcular cada centroide como la media de los puntos asignados:
   $$\mu_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} x_i$$
4. **Convergencia:** repetir pasos 2 y 3 hasta que los centroides no cambien (o el cambio sea menor a un umbral $\epsilon$).

**Complejidad:** $O(n \cdot k \cdot d \cdot I)$ donde $n$ = puntos, $k$ = clusters, $d$ = dimensiones, $I$ = iteraciones.

### Inicialización K-means++
La inicialización aleatoria puede converger a mínimos locales subóptimos. **K-means++** mejora esto seleccionando centroides iniciales separados entre sí:
1. Elegir el primer centroide uniformemente al azar.
2. Para cada centroide siguiente, seleccionar un punto con probabilidad proporcional a $d(x)^2$ (distancia al centroide más cercano ya elegido).

Esto garantiza mejores resultados en promedio. **sklearn usa K-means++ por defecto** (`init='k-means++'`).

In [ ]:
# ── Generamos datos con clusters bien definidos ───────────────────────────────
X_blobs, y_true = make_blobs(
    n_samples=500,
    centers=4,
    cluster_std=0.8,
    random_state=42
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Datos sin etiquetar (lo que ve el algoritmo)
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], alpha=0.6, s=30, color='steelblue')
axes[0].set_title('Datos sin etiquetar (entrada al algoritmo)', fontsize=13)
axes[0].set_xlabel('Característica 1')
axes[0].set_ylabel('Característica 2')

# Datos con etiquetas reales
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
for c in range(4):
    mask = y_true == c
    axes[1].scatter(X_blobs[mask, 0], X_blobs[mask, 1],
                    color=colors[c], alpha=0.6, s=30, label=f'Cluster {c+1}')
axes[1].set_title('Agrupación real (referencia)', fontsize=13)
axes[1].set_xlabel('Característica 1')
axes[1].legend()

plt.suptitle('make_blobs — Dataset de ejemplo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualización del proceso iterativo de K-means ────────────────────────────
# Simulamos las primeras iteraciones manualmente para ilustrar el algoritmo

def plot_kmeans_step(X, centroids, labels=None, step=0, ax=None):
    """Visualiza un paso del algoritmo K-means."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    
    cmap = plt.cm.Set1
    k = len(centroids)
    
    if labels is not None:
        # Paso de asignación: colorear puntos según cluster asignado
        for c in range(k):
            mask = labels == c
            ax.scatter(X[mask, 0], X[mask, 1],
                       color=cmap(c / k), alpha=0.5, s=20)
    else:
        ax.scatter(X[:, 0], X[:, 1], color='lightgray', alpha=0.5, s=20)
    
    # Dibujar centroides
    for c, centroid in enumerate(centroids):
        ax.scatter(*centroid, s=250, color=cmap(c / k),
                   marker='*', edgecolors='black', linewidths=1.5, zorder=5)
    
    ax.set_title(f'Iteración {step}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


# Inicialización manual de 4 centroides aleatorios
rng = np.random.RandomState(0)
k = 4
init_centroids = X_blobs[rng.choice(len(X_blobs), k, replace=False)]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

centroids = init_centroids.copy()
labels = None

for step in range(6):
    ax = axes[step]
    
    if step == 0:
        # Mostrar centroides iniciales sin asignación
        plot_kmeans_step(X_blobs, centroids, labels=None, step=0, ax=ax)
        ax.set_title('Paso 0: Centroides iniciales (aleatorios)', fontsize=10)
    else:
        # Paso de asignación
        dists = np.array([[np.linalg.norm(x - c) for c in centroids] for x in X_blobs])
        labels = np.argmin(dists, axis=1)
        plot_kmeans_step(X_blobs, centroids, labels=labels, step=step, ax=ax)
        ax.set_title(f'Iteración {step}: Asignación y actualización', fontsize=10)
        
        # Paso de actualización de centroides
        centroids = np.array([X_blobs[labels == c].mean(axis=0) for c in range(k)])

plt.suptitle('Algoritmo K-means: Evolución iterativa', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Implementación con sklearn

In [ ]:
# ── K-means con sklearn ───────────────────────────────────────────────────────
kmeans = KMeans(
    n_clusters=4,        # número de clusters k
    init='k-means++',    # inicialización inteligente (K-means++)
    n_init=10,           # número de ejecuciones con diferentes inicializaciones
    max_iter=300,        # máximo de iteraciones por ejecución
    random_state=42
)

kmeans.fit(X_blobs)

# Resultados principales
labels_km = kmeans.labels_
centroids_km = kmeans.cluster_centers_

print('=== Resultados de K-means ===')
print(f'  Inercia (WCSS):          {kmeans.inertia_:.2f}')
print(f'  Número de iteraciones:   {kmeans.n_iter_}')
print(f'  Centroides encontrados:\n{centroids_km.round(3)}')

# Silhouette score
sil = silhouette_score(X_blobs, labels_km)
print(f'  Silhouette score:        {sil:.4f}')

In [ ]:
# ── Visualización del resultado de K-means ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

cmap = plt.cm.Set1
for c in range(4):
    mask = labels_km == c
    ax.scatter(X_blobs[mask, 0], X_blobs[mask, 1],
               color=cmap(c / 4), alpha=0.6, s=30, label=f'Cluster {c+1}')

# Dibujar centroides
ax.scatter(centroids_km[:, 0], centroids_km[:, 1],
           s=300, color='gold', marker='*',
           edgecolors='black', linewidths=1.5,
           zorder=5, label='Centroides')

ax.set_title(f'K-means (k=4) — Silhouette: {sil:.3f}', fontsize=13)
ax.set_xlabel('Característica 1')
ax.set_ylabel('Característica 2')
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 Selección de k: Método del Codo (Elbow Method)

El problema fundamental de K-means es que requiere especificar $k$ de antemano. El **método del codo** ayuda a encontrar un $k$ razonable:

- Se entrena K-means para distintos valores de $k$ y se grafica la **inercia** vs. $k$.
- La inercia siempre disminuye al aumentar $k$ (con $k = n$, cada punto es su propio cluster y la inercia es 0).
- El "codo" es el punto donde la reducción de inercia se aplana: agregar más clusters da pocos beneficios.

> **Limitación:** el codo puede no ser siempre evidente. Se recomienda combinar con el Silhouette score.

In [ ]:
# ── Método del Codo + Silhouette score ───────────────────────────────────────
k_range = range(2, 11)  # probar k de 2 a 10
inercias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_blobs)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, km.labels_))

# ── Gráfico combinado ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Codo
axes[0].plot(list(k_range), inercias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', linewidth=1.5, label='k óptimo = 4')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia (WCSS)')
axes[0].set_title('Método del Codo', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette
axes[1].plot(list(k_range), silhouettes, 'rs-', linewidth=2, markersize=8)
axes[1].axvline(x=4, color='blue', linestyle='--', linewidth=1.5, label='k óptimo = 4')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Coeficiente de Silhouette por k', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Selección del número óptimo de clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla resumen
df_metrics = pd.DataFrame({
    'k': list(k_range),
    'Inercia': [f'{v:.1f}' for v in inercias],
    'Silhouette': [f'{v:.4f}' for v in silhouettes]
})
print(df_metrics.to_string(index=False))

### 2.4 Análisis de Silhouette por muestra

El **diagrama de Silhouette** muestra el coeficiente de silhouette de cada muestra, agrupadas por cluster. Permite:
- Detectar clusters de tamaños muy desiguales.
- Identificar puntos "mal asignados" (coeficiente negativo).
- Comparar la calidad relativa entre clusters.

In [ ]:
# ── Diagrama de Silhouette ────────────────────────────────────────────────────
def plot_silhouette(X, labels, k, ax, title=''):
    """Genera el diagrama de Silhouette para un resultado de clustering."""
    sil_avg = silhouette_score(X, labels)
    sil_vals = silhouette_samples(X, labels)
    
    y_lower = 10
    cmap = plt.cm.nipy_spectral
    
    for c in range(k):
        # Silhouette de las muestras del cluster c, ordenadas
        cluster_sil = np.sort(sil_vals[labels == c])
        size_c = cluster_sil.shape[0]
        y_upper = y_lower + size_c
        
        color = cmap(c / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                         0, cluster_sil,
                         facecolor=color, edgecolor=color, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * size_c, str(c + 1))
        y_lower = y_upper + 10
    
    # Línea de silhouette promedio
    ax.axvline(x=sil_avg, color='red', linestyle='--', linewidth=1.5)
    ax.text(sil_avg + 0.01, ax.get_ylim()[1] * 0.95,
            f'Promedio = {sil_avg:.3f}', color='red', fontsize=9)
    
    ax.set_xlabel('Coeficiente de Silhouette')
    ax.set_ylabel('Cluster')
    ax.set_title(title, fontsize=11)
    ax.set_xlim([-0.2, 1.1])
    ax.set_yticks([])


# Comparar k=3, k=4, k=5
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for i, k in enumerate([3, 4, 5]):
    km_temp = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels_temp = km_temp.fit_predict(X_blobs)
    plot_silhouette(X_blobs, labels_temp, k, axes[i], title=f'k = {k}')

plt.suptitle('Diagrama de Silhouette para distintos valores de k', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5 Limitaciones de K-means

K-means tiene suposiciones implícitas que lo hacen inadecuado para ciertos tipos de datos:

| Limitación | Causa | Solución alternativa |
|-----------|-------|---------------------|
| Clusters no convexos | Asigna por distancia euclidiana | DBSCAN, Spectral Clustering |
| Sensibilidad a outliers | Los outliers desplazan los centroides | DBSCAN, K-medoids |
| Requiere especificar k | No hay mecanismo automático | Método del codo, Silhouette, BIC |
| Clusters de tamaños muy diferentes | Igual peso a todos los clusters | Gaussian Mixture Models |
| Clusters de varianzas diferentes | Asume clusters esféricos | GMM con covarianza completa |

In [ ]:
# ── Demostración de limitaciones de K-means ───────────────────────────────────
# Generamos tres tipos de datasets donde K-means falla
n = 400
X_moons, _   = make_moons(n_samples=n, noise=0.08, random_state=42)
X_circles, _ = make_circles(n_samples=n, factor=0.5, noise=0.05, random_state=42)
X_aniso, y_a = make_blobs(n_samples=n, random_state=42)
# Transformación anisotrópica: clusters elongados
transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
X_aniso = X_aniso @ transform

datasets = [
    (X_moons,   'Lunas (make_moons)'),
    (X_circles, 'Círculos concéntricos (make_circles)'),
    (X_aniso,   'Clusters anisotrópicos'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (X_data, title) in zip(axes, datasets):
    km_fail = KMeans(n_clusters=2, n_init=10, random_state=42)
    labels_fail = km_fail.fit_predict(X_data)
    
    for c in range(2):
        mask = labels_fail == c
        ax.scatter(X_data[mask, 0], X_data[mask, 1],
                   alpha=0.6, s=25, label=f'Cluster {c+1}')
    
    cx = km_fail.cluster_centers_
    ax.scatter(cx[:, 0], cx[:, 1], s=200, marker='*', color='gold',
               edgecolors='black', zorder=5)
    ax.set_title(f'K-means falla: {title}', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Limitaciones de K-means en geometrías no convexas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. DBSCAN — Density-Based Spatial Clustering of Applications with Noise

### 3.1 Motivación y teoría

**DBSCAN** es un algoritmo basado en densidad que supera las principales limitaciones de K-means:
- Detecta clusters de **forma arbitraria**.
- Es **robusto a outliers** (los clasifica como ruido).
- **No requiere especificar k** de antemano.

### Parámetros fundamentales

| Parámetro | Descripción | Efecto |
|-----------|-------------|--------|
| `eps` ($\varepsilon$) | Radio de vecindad | Mayor $\varepsilon$ → más puntos juntos → clusters más grandes |
| `min_samples` | Mínimo de vecinos para ser núcleo | Mayor → más restrictivo → más ruido |

### Tipos de puntos

$$\text{Vecindad de } p: N_{\varepsilon}(p) = \{q \in D : d(p, q) \leq \varepsilon\}$$

- **Punto núcleo (Core point):** $|N_{\varepsilon}(p)| \geq \text{min\_samples}$
- **Punto frontera (Border point):** no es núcleo, pero está en la vecindad de un núcleo.
- **Punto ruido (Noise point):** no es núcleo ni frontera. Outlier.

### Algoritmo
1. Para cada punto no visitado, calcular su vecindad.
2. Si es punto núcleo, crear un nuevo cluster y expandirlo recursivamente.
3. Si no es punto núcleo, marcarlo como frontera o ruido.
4. Continuar hasta visitar todos los puntos.

In [ ]:
# ── Visualización conceptual: tipos de puntos en DBSCAN ──────────────────────
# Creamos un ejemplo pequeño para ilustrar los conceptos
np.random.seed(7)

# Cluster denso
X_demo = np.vstack([
    np.random.randn(30, 2) * 0.4 + [0, 0],       # cluster 1
    np.random.randn(25, 2) * 0.35 + [3, 1],      # cluster 2
    np.array([[1.5, 3.5], [-1, 2], [5, -1]])      # ruido
])

eps_demo = 0.6
min_samples_demo = 5

db_demo = DBSCAN(eps=eps_demo, min_samples=min_samples_demo)
labels_demo = db_demo.fit_predict(X_demo)

# Identificar tipos de puntos
core_mask = np.zeros(len(X_demo), dtype=bool)
core_mask[db_demo.core_sample_indices_] = True
noise_mask = labels_demo == -1
border_mask = ~core_mask & ~noise_mask

fig, ax = plt.subplots(figsize=(9, 6))

# Puntos núcleo
ax.scatter(X_demo[core_mask & (labels_demo == 0), 0],
           X_demo[core_mask & (labels_demo == 0), 1],
           s=80, color='#2196F3', label='Núcleo — Cluster 1', zorder=3)
ax.scatter(X_demo[core_mask & (labels_demo == 1), 0],
           X_demo[core_mask & (labels_demo == 1), 1],
           s=80, color='#4CAF50', label='Núcleo — Cluster 2', zorder=3)

# Puntos frontera
if border_mask.any():
    ax.scatter(X_demo[border_mask, 0], X_demo[border_mask, 1],
               s=80, color='orange', label='Frontera', marker='D', zorder=3)

# Ruido
ax.scatter(X_demo[noise_mask, 0], X_demo[noise_mask, 1],
           s=100, color='red', label='Ruido (outlier)', marker='x',
           linewidths=2, zorder=4)

# Dibujar vecindad epsilon de un punto núcleo
core_indices = db_demo.core_sample_indices_
p_example = X_demo[core_indices[0]]
circle = plt.Circle(p_example, eps_demo, fill=False,
                    color='purple', linestyle='--', linewidth=1.5, label=f'ε = {eps_demo}')
ax.add_patch(circle)
ax.plot(*p_example, 'p', color='purple', markersize=15, zorder=5)

ax.set_title(f'DBSCAN — Tipos de puntos (ε={eps_demo}, min_samples={min_samples_demo})', fontsize=13)
ax.set_xlabel('Característica 1')
ax.set_ylabel('Característica 2')
ax.legend(loc='upper right', fontsize=9)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f'Puntos núcleo:    {core_mask.sum()}')
print(f'Puntos frontera:  {border_mask.sum()}')
print(f'Puntos ruido:     {noise_mask.sum()}')

### 3.2 Selección de epsilon: Gráfica k-distancia

El parámetro más crítico de DBSCAN es `eps`. La **gráfica k-distancia** ayuda a encontrar un valor apropiado:

1. Para cada punto, calcular la distancia a su k-ésimo vecino más cercano (usando `k = min_samples - 1`).
2. Ordenar estas distancias de mayor a menor.
3. El valor de `eps` óptimo está en el **codo** de esta gráfica: separa los puntos "densos" (que tienen vecinos cercanos) de los puntos de ruido (cuyos vecinos están lejos).

In [ ]:
# ── Gráfica k-distancia para selección de epsilon ────────────────────────────
def plot_k_distance(X, min_samples=5, ax=None):
    """
    Calcula y grafica la distancia al k-ésimo vecino más cercano.
    El 'codo' sugiere el valor óptimo de epsilon.
    """
    k = min_samples  # usar min_samples como k
    nbrs = NearestNeighbors(n_neighbors=k).fit(X)
    distances, _ = nbrs.kneighbors(X)
    
    # Distancia al k-ésimo vecino (última columna)
    k_distances = np.sort(distances[:, -1])[::-1]
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(k_distances, linewidth=2, color='steelblue')
    ax.set_xlabel(f'Puntos ordenados por distancia al {k}-ésimo vecino')
    ax.set_ylabel(f'Distancia al {k}-ésimo vecino (ε candidato)')
    ax.set_title(f'Gráfica k-distancia (k={k}) — Buscar el "codo"', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    return k_distances


# Aplicar a los datos de lunas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

k_dists_moons   = plot_k_distance(X_moons, min_samples=5, ax=axes[0])
axes[0].axhline(y=0.25, color='red', linestyle='--', linewidth=1.5, label='ε sugerido ≈ 0.25')
axes[0].set_title('k-distancia: make_moons', fontsize=12)
axes[0].legend()

k_dists_circles = plot_k_distance(X_circles, min_samples=5, ax=axes[1])
axes[1].axhline(y=0.15, color='red', linestyle='--', linewidth=1.5, label='ε sugerido ≈ 0.15')
axes[1].set_title('k-distancia: make_circles', fontsize=12)
axes[1].legend()

plt.suptitle('Gráfica k-distancia para seleccionar ε en DBSCAN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 DBSCAN vs K-means en geometrías complejas

In [ ]:
# ── Comparación DBSCAN vs K-means en make_moons y make_circles ───────────────
def plot_clustering_comparison(X, labels_km, labels_db, dataset_name, axes_row):
    """Muestra K-means y DBSCAN lado a lado para un dataset."""
    
    for ax, labels, title in zip(
        axes_row,
        [labels_km, labels_db],
        ['K-means (k=2)', 'DBSCAN']
    ):
        unique_labels = set(labels)
        cmap = plt.cm.Set1
        n_clusters = len([l for l in unique_labels if l != -1])
        
        for i, label in enumerate(sorted(unique_labels)):
            mask = labels == label
            if label == -1:
                # Ruido en DBSCAN
                ax.scatter(X[mask, 0], X[mask, 1],
                           color='black', s=15, alpha=0.5,
                           marker='x', label='Ruido')
            else:
                ax.scatter(X[mask, 0], X[mask, 1],
                           color=cmap(i / max(n_clusters, 1)),
                           s=25, alpha=0.7, label=f'Cluster {label+1}')
        
        ax.set_title(f'{dataset_name}\n{title} — {n_clusters} cluster(s)', fontsize=11)
        ax.legend(fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])


fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# -- make_moons --
km_moons = KMeans(n_clusters=2, n_init=10, random_state=42)
labels_km_moons = km_moons.fit_predict(X_moons)

db_moons = DBSCAN(eps=0.25, min_samples=5)
labels_db_moons = db_moons.fit_predict(X_moons)

plot_clustering_comparison(X_moons, labels_km_moons, labels_db_moons,
                           'make_moons', axes[0])

# -- make_circles --
km_circles = KMeans(n_clusters=2, n_init=10, random_state=42)
labels_km_circles = km_circles.fit_predict(X_circles)

db_circles = DBSCAN(eps=0.15, min_samples=5)
labels_db_circles = db_circles.fit_predict(X_circles)

plot_clustering_comparison(X_circles, labels_km_circles, labels_db_circles,
                           'make_circles', axes[1])

plt.suptitle('K-means vs DBSCAN en geometrías no convexas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Reporte de calidad
print('=== Comparación de calidad (Silhouette) ===')
for name, X_d, lkm, ldb in [
    ('make_moons',   X_moons,   labels_km_moons,   labels_db_moons),
    ('make_circles', X_circles, labels_km_circles, labels_db_circles),
]:
    sil_km = silhouette_score(X_d, lkm)
    # DBSCAN puede tener ruido (-1); calcular solo en puntos asignados
    mask_assigned = ldb != -1
    sil_db = silhouette_score(X_d[mask_assigned], ldb[mask_assigned]) if mask_assigned.sum() > 1 else float('nan')
    n_noise = (ldb == -1).sum()
    print(f'  {name}:')
    print(f'    K-means Silhouette:  {sil_km:.4f}')
    print(f'    DBSCAN  Silhouette:  {sil_db:.4f}  (ruido: {n_noise} puntos)')

### 3.4 Sensibilidad de DBSCAN a los hiperparámetros

In [ ]:
# ── Efecto de variar epsilon en DBSCAN ───────────────────────────────────────
eps_values = [0.1, 0.2, 0.35, 0.6]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, eps_val in zip(axes, eps_values):
    db_temp = DBSCAN(eps=eps_val, min_samples=5)
    labels_temp = db_temp.fit_predict(X_moons)
    
    unique = set(labels_temp)
    n_clusters = len([l for l in unique if l != -1])
    n_noise = (labels_temp == -1).sum()
    
    cmap = plt.cm.Set1
    for label in sorted(unique):
        mask = labels_temp == label
        if label == -1:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                       color='black', s=15, marker='x', alpha=0.6)
        else:
            ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                       color=cmap(label / max(n_clusters, 1)), s=20, alpha=0.7)
    
    ax.set_title(f'ε = {eps_val}\n{n_clusters} cluster(s), {n_noise} ruido', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('DBSCAN (make_moons): efecto de variar ε (min_samples=5)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Clustering Jerárquico (Aglomerativo)

### 4.1 Motivación y tipos

El **clustering jerárquico** construye una jerarquía de clusters representada en un **dendrograma**. Existen dos enfoques:

- **Aglomerativo (bottom-up):** comienza con cada punto como su propio cluster y los va uniendo.
- **Divisivo (top-down):** comienza con un único cluster y lo divide sucesivamente.

Sklearn implementa el método **aglomerativo**, que es el más común.

### Ventajas sobre K-means
- No requiere especificar $k$ de antemano (se elige al cortar el dendrograma).
- Produce una jerarquía de clusters a distintos niveles de granularidad.
- Funciona con cualquier métrica de distancia.

### 4.2 Métodos de enlace (Linkage)

El **criterio de enlace** define cómo se mide la distancia entre dos clusters $C_i$ y $C_j$:

| Método | Fórmula | Característica |
|--------|---------|----------------|
| **Single** | $\min_{a \in C_i, b \in C_j} d(a, b)$ | Encadenamiento, clusters elongados |
| **Complete** | $\max_{a \in C_i, b \in C_j} d(a, b)$ | Clusters compactos |
| **Average** | $\frac{1}{|C_i||C_j|}\sum d(a,b)$ | Balance entre single y complete |
| **Ward** | Minimiza la varianza total intracluster | Clusters esféricos y similares en tamaño |

In [ ]:
# ── Dendrograma con scipy ─────────────────────────────────────────────────────
# Usamos un subconjunto para que el dendrograma sea legible
np.random.seed(42)
X_hier, _ = make_blobs(n_samples=50, centers=3, cluster_std=0.6, random_state=42)

linkage_methods = ['single', 'complete', 'average', 'ward']
method_titles = {
    'single': 'Single (vecino más cercano)',
    'complete': 'Complete (vecino más lejano)',
    'average': 'Average (promedio)',
    'ward': 'Ward (mínima varianza)'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, method in zip(axes, linkage_methods):
    # Calcular matriz de enlace
    Z = linkage(X_hier, method=method)
    
    dendrogram(
        Z,
        ax=ax,
        truncate_mode='lastp',   # mostrar solo las últimas p fusiones
        p=20,
        show_leaf_counts=True,
        leaf_rotation=45,
        color_threshold=0.7 * max(Z[:, 2])  # colorear clusters
    )
    ax.set_title(f'Enlace: {method_titles[method]}', fontsize=12)
    ax.set_xlabel('Muestra / Cluster', fontsize=10)
    ax.set_ylabel('Distancia de fusión', fontsize=10)
    ax.grid(True, alpha=0.2)

plt.suptitle('Dendrogramas con distintos métodos de enlace', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Corte del dendrograma para obtener clusters

Para obtener un número específico de clusters, se **corta el dendrograma** a una cierta altura. La elección de la altura de corte es equivalente a elegir $k$:
- Corte alto → pocos clusters grandes.
- Corte bajo → muchos clusters pequeños.

**Regla práctica:** buscar el nivel donde las fusiones más largas se realizan (grandes saltos en el eje Y del dendrograma).

In [ ]:
# ── Corte del dendrograma y visualización de clusters ─────────────────────────
Z_ward = linkage(X_hier, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Dendrograma con línea de corte
threshold = 5.0  # umbral de corte
dendrogram(
    Z_ward,
    ax=axes[0],
    truncate_mode='lastp',
    p=20,
    show_leaf_counts=True,
    leaf_rotation=45,
    color_threshold=threshold
)
axes[0].axhline(y=threshold, color='red', linestyle='--', linewidth=2,
                label=f'Corte en {threshold}')
axes[0].set_title('Dendrograma Ward — Línea de corte', fontsize=12)
axes[0].set_xlabel('Muestra / Cluster')
axes[0].set_ylabel('Distancia de fusión')
axes[0].legend()

# Resultado del clustering
# sklearn AgglomerativeClustering para obtener etiquetas
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_hier)

cmap = plt.cm.Set1
for c in range(3):
    mask = labels_agg == c
    axes[1].scatter(X_hier[mask, 0], X_hier[mask, 1],
                    color=cmap(c / 3), s=60, alpha=0.8, label=f'Cluster {c+1}')

sil_agg = silhouette_score(X_hier, labels_agg)
axes[1].set_title(f'Clusters resultantes (k=3, Ward)\nSilhouette: {sil_agg:.3f}', fontsize=12)
axes[1].set_xlabel('Característica 1')
axes[1].set_ylabel('Característica 2')
axes[1].legend()

plt.suptitle('Clustering Jerárquico Aglomerativo — Ward', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Comparación de los 4 métodos de enlace en los datos ──────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, method in zip(axes, linkage_methods):
    agg_temp = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels_temp = agg_temp.fit_predict(X_hier)
    sil_temp = silhouette_score(X_hier, labels_temp)
    
    for c in range(3):
        mask = labels_temp == c
        ax.scatter(X_hier[mask, 0], X_hier[mask, 1],
                   color=cmap(c / 3), s=50, alpha=0.8, label=f'C{c+1}')
    
    ax.set_title(f'{method_titles[method]}\nSilhouette: {sil_temp:.3f}', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Comparación de métodos de enlace (k=3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Comparación de Métodos

### 5.1 Resumen teórico

| Característica | K-means | DBSCAN | Jerárquico (Ward) |
|---------------|---------|--------|-------------------|
| Requiere especificar $k$ | Sí | No | Sí (en el corte) |
| Manejo de outliers | No | Sí (ruido) | Parcial |
| Forma de clusters | Solo convexos | Arbitraria | Convexa (Ward) |
| Escala con $n$ | $O(nkd)$ — rápido | $O(n^2)$ — lento | $O(n^2 \log n)$ — lento |
| Hiperparámetros | $k$ | $\varepsilon$, `min_samples` | $k$ o altura de corte, enlace |
| Dendrograma | No | No | Sí |
| Clusters vacíos | Posible | No aplica | No |
| Recomendado cuando | Clusters esféricos, $n$ grande | Datos con ruido, forma irregular | Se necesita jerarquía |

### 5.2 Comparación visual en múltiples geometrías

In [ ]:
# ── Benchmark visual: los tres algoritmos en cinco datasets ───────────────────
np.random.seed(42)

# Datasets de prueba
n_pts = 300
datasets_bench = [
    (make_blobs(n_samples=n_pts, centers=3, cluster_std=0.7, random_state=42)[0], 'Blobs'),
    (make_moons(n_samples=n_pts, noise=0.07, random_state=42)[0], 'Lunas'),
    (make_circles(n_samples=n_pts, factor=0.4, noise=0.06, random_state=42)[0], 'Círculos'),
]

algorithms = [
    ('K-means',  lambda X: KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X)),
    ('DBSCAN',   lambda X: DBSCAN(eps=0.3, min_samples=5).fit_predict(X)),
    ('Jerárquico (Ward)', lambda X: AgglomerativeClustering(n_clusters=2, linkage='ward').fit_predict(X)),
]

fig, axes = plt.subplots(len(datasets_bench), len(algorithms),
                         figsize=(13, 11))

for row, (X_b, ds_name) in enumerate(datasets_bench):
    # Escalar datos
    X_scaled = StandardScaler().fit_transform(X_b)
    
    for col, (alg_name, alg_fn) in enumerate(algorithms):
        ax = axes[row][col]
        labels_b = alg_fn(X_scaled)
        
        unique_b = sorted(set(labels_b))
        n_cl = len([l for l in unique_b if l != -1])
        cmap_b = plt.cm.Set1
        
        color_idx = 0
        for label in unique_b:
            mask = labels_b == label
            if label == -1:
                ax.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                           color='k', s=12, marker='x', alpha=0.5)
            else:
                ax.scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                           color=cmap_b(color_idx / max(n_cl, 1)),
                           s=15, alpha=0.7)
                color_idx += 1
        
        # Calcular silhouette solo si hay más de 1 cluster
        mask_valid = labels_b != -1
        if len(set(labels_b[mask_valid])) > 1 and mask_valid.sum() > 1:
            sil_b = silhouette_score(X_scaled[mask_valid], labels_b[mask_valid])
            sil_txt = f'Sil={sil_b:.3f}'
        else:
            sil_txt = 'N/A'
        
        ax.set_title(f'{ds_name} — {alg_name}\n{sil_txt}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Comparación de algoritmos en distintas geometrías de datos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Caso Práctico — Dataset Housing de California

Aplicamos los tres algoritmos a datos reales de viviendas. El objetivo es identificar segmentos geográficos o socioeconómicos en el mercado inmobiliario.

In [ ]:
# ── Carga de datos ────────────────────────────────────────────────────────────
import os

# Ruta principal (relativa al notebook de Semana 9)
housing_path = '../Semana 8 - Aprendizaje no Supervisado - Agrupación/housing.csv'

if os.path.exists(housing_path):
    housing = pd.read_csv(housing_path)
    print('Archivo housing.csv cargado correctamente.')
    data_source = 'housing.csv'
else:
    # Fallback: usar California Housing de sklearn
    print('housing.csv no encontrado. Usando sklearn.datasets.fetch_california_housing.')
    from sklearn.datasets import fetch_california_housing
    cal = fetch_california_housing()
    housing = pd.DataFrame(cal.data, columns=cal.feature_names)
    housing['median_house_value'] = cal.target * 100_000  # escalar a dólares
    data_source = 'California Housing (sklearn)'

print(f'\nFuente: {data_source}')
print(f'Dimensiones: {housing.shape}')
print(f'\nColumnas: {list(housing.columns)}')
housing.head()

In [ ]:
# ── Exploración inicial ───────────────────────────────────────────────────────
print('=== Estadísticas descriptivas ===')
print(housing.describe().round(2))

print('\n=== Valores faltantes ===')
print(housing.isnull().sum())

In [ ]:
# ── Selección y preprocesamiento de características ───────────────────────────

# Seleccionar características numéricas relevantes
# Intentar columnas comunes; adaptar según el dataset disponible
candidate_cols = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',  # sklearn
    'median_income', 'housing_median_age', 'total_rooms', 'total_bedrooms',      # housing.csv
    'population', 'households', 'median_house_value'                             # housing.csv
]

# Usar solo las columnas que existen en el dataset
feature_cols = [c for c in candidate_cols if c in housing.columns]

# Si tenemos latitud/longitud, añadirlas para clustering geográfico
for geo_col in ['latitude', 'longitude', 'Latitude', 'Longitude']:
    if geo_col in housing.columns and geo_col not in feature_cols:
        feature_cols.append(geo_col)

print(f'Características seleccionadas: {feature_cols}')

# Eliminar filas con valores faltantes en esas columnas
df_clean = housing[feature_cols].dropna()
print(f'Filas después de limpiar: {len(df_clean)} / {len(housing)}')

# Muestra aleatoria para eficiencia (máx. 3000 filas)
if len(df_clean) > 3000:
    df_sample = df_clean.sample(3000, random_state=42).reset_index(drop=True)
else:
    df_sample = df_clean.reset_index(drop=True)

# Escalar características
scaler = StandardScaler()
X_housing = scaler.fit_transform(df_sample)
print(f'\nMatriz de características estandarizada: {X_housing.shape}')

In [ ]:
# ── Selección de k para K-means en datos Housing ─────────────────────────────
k_range_h = range(2, 9)
inercias_h = []
silhouettes_h = []

for k in k_range_h:
    km_h = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=42)
    labels_h = km_h.fit_predict(X_housing)
    inercias_h.append(km_h.inertia_)
    silhouettes_h.append(silhouette_score(X_housing, labels_h))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(k_range_h), inercias_h, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo — Housing', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range_h), silhouettes_h, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette por k — Housing', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Marcar k óptimo según silhouette
k_opt = list(k_range_h)[np.argmax(silhouettes_h)]
axes[1].axvline(x=k_opt, color='blue', linestyle='--',
                label=f'k óptimo = {k_opt}')
axes[1].legend()

plt.suptitle('Selección de k — Dataset Housing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'k óptimo según Silhouette: {k_opt}')

In [ ]:
# ── Aplicación de K-means óptimo ─────────────────────────────────────────────
km_final = KMeans(n_clusters=k_opt, init='k-means++', n_init=10, random_state=42)
df_sample['cluster_kmeans'] = km_final.fit_predict(X_housing)

# Perfil de cada cluster
print(f'=== Perfil de clusters (K-means, k={k_opt}) ===')
print(df_sample.groupby('cluster_kmeans')[feature_cols[:5]].mean().round(3))

In [ ]:
# ── Visualización geográfica (si hay latitud/longitud) ────────────────────────
# Verificar columnas geográficas
lat_col = next((c for c in df_sample.columns if c.lower() == 'latitude'), None)
lon_col = next((c for c in df_sample.columns if c.lower() == 'longitude'), None)

if lat_col and lon_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Mapa de clusters K-means
    for c in range(k_opt):
        mask = df_sample['cluster_kmeans'] == c
        axes[0].scatter(
            df_sample.loc[mask, lon_col],
            df_sample.loc[mask, lat_col],
            s=5, alpha=0.5, label=f'Cluster {c+1}'
        )
    axes[0].set_title(f'K-means (k={k_opt}) — Distribución geográfica', fontsize=12)
    axes[0].set_xlabel('Longitud')
    axes[0].set_ylabel('Latitud')
    axes[0].legend(markerscale=3, fontsize=9)
    
    # Heatmap de características por cluster
    numeric_features = [c for c in feature_cols if c not in [lat_col, lon_col]]
    if numeric_features:
        cluster_means = df_sample.groupby('cluster_kmeans')[numeric_features[:6]].mean()
        # Normalizar para heatmap
        cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())
        im = axes[1].imshow(cluster_means_norm.values, cmap='YlOrRd', aspect='auto')
        axes[1].set_xticks(range(len(cluster_means_norm.columns)))
        axes[1].set_xticklabels(cluster_means_norm.columns, rotation=30, ha='right', fontsize=9)
        axes[1].set_yticks(range(k_opt))
        axes[1].set_yticklabels([f'Cluster {c+1}' for c in range(k_opt)])
        axes[1].set_title('Perfil de clusters (valores normalizados)', fontsize=12)
        plt.colorbar(im, ax=axes[1])
    
    plt.suptitle('Análisis de Clusters — Housing Dataset', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    # Sin coordenadas: usar PCA para visualización 2D
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_housing)
    
    fig, ax = plt.subplots(figsize=(9, 6))
    for c in range(k_opt):
        mask = df_sample['cluster_kmeans'].values == c
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   s=15, alpha=0.6, label=f'Cluster {c+1}')
    ax.set_title(f'K-means (k={k_opt}) — Proyección PCA 2D', fontsize=13)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Aplicación de Clustering Jerárquico y comparación ────────────────────────
# Jerárquico con Ward (solo viable en muestras menores)
n_hier = min(1000, len(df_sample))  # limitar para eficiencia
X_hier_housing = X_housing[:n_hier]

agg_housing = AgglomerativeClustering(n_clusters=k_opt, linkage='ward')
labels_agg_housing = agg_housing.fit_predict(X_hier_housing)

# Dendrograma
Z_housing = linkage(X_hier_housing[:100], method='ward')  # solo 100 para visualización

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(
    Z_housing,
    ax=ax,
    truncate_mode='lastp',
    p=20,
    leaf_rotation=45,
    color_threshold=0.6 * max(Z_housing[:, 2])
)
ax.set_title('Dendrograma Ward — Housing (muestra de 100)', fontsize=13)
ax.set_xlabel('Instancias')
ax.set_ylabel('Distancia de fusión')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

# Silhouette comparativo
sil_km_h    = silhouette_score(X_housing, df_sample['cluster_kmeans'].values)
sil_agg_h   = silhouette_score(X_hier_housing, labels_agg_housing)

print('=== Comparación final en Housing ===')
print(f'  K-means (k={k_opt})          Silhouette: {sil_km_h:.4f}')
print(f'  Jerárquico Ward (k={k_opt})   Silhouette: {sil_agg_h:.4f}  (n={n_hier})')

In [ ]:
# ── Resumen estadístico de clusters K-means ───────────────────────────────────
print('=== Análisis descriptivo por cluster (K-means) ===')

numeric_cols = [c for c in feature_cols if c in df_sample.columns]
cluster_stats = df_sample.groupby('cluster_kmeans')[numeric_cols[:6]].agg(['mean', 'std'])
print(cluster_stats.round(3))

print('\n=== Tamaño de cada cluster ===')
print(df_sample['cluster_kmeans'].value_counts().sort_index())

---
## 7. Guía de Selección de Algoritmo

El siguiente diagrama de decisión resume cuándo usar cada algoritmo:

```
¿Necesita manejar ruido/outliers explícitamente?
├── Sí → DBSCAN
│         ├── Clusters de forma irregular → DBSCAN es ideal
│         └── Ajustar ε con la gráfica k-distancia
│
└── No → ¿Necesita una jerarquía de clusters o dendrograma?
          ├── Sí → Clustering Jerárquico (Ward para datos continuos)
          │         └── Útil cuando no se sabe k de antemano (inspección visual)
          │
          └── No → ¿Los clusters son aproximadamente esféricos y el dataset es grande?
                    ├── Sí → K-means (rápido, escalable)
                    │         └── Usar método del codo + Silhouette para elegir k
                    └── No → Considerar Gaussian Mixture Models o Spectral Clustering
```

### Buenas prácticas generales

1. **Siempre escalar los datos** antes de clustering (excepción: datos de texto con TF-IDF ya normalizado).
2. **Combinar métricas:** nunca confiar en una sola métrica de calidad. Usar Silhouette + visualización + conocimiento del dominio.
3. **Experimentar con múltiples algoritmos** y comparar resultados en el contexto del problema.
4. **Reducir dimensionalidad** (PCA, UMAP) antes de clustering si $d > 10$ para evitar la maldición de la dimensionalidad.
5. **Interpretar los clusters** en términos del negocio o dominio: ¿tienen sentido los grupos encontrados?

In [ ]:
# ── Tabla comparativa final con todos los datasets y algoritmos ───────────────
results = []

# Datos esféricos (blobs)
X_b, _ = make_blobs(n_samples=500, centers=3, cluster_std=0.8, random_state=42)
X_b_sc = StandardScaler().fit_transform(X_b)

# Datos no convexos (moons)
X_m, _ = make_moons(n_samples=500, noise=0.07, random_state=42)
X_m_sc = StandardScaler().fit_transform(X_m)

configs = [
    ('Blobs',  X_b_sc, KMeans(n_clusters=3, n_init=10, random_state=42)),
    ('Blobs',  X_b_sc, DBSCAN(eps=0.5, min_samples=5)),
    ('Blobs',  X_b_sc, AgglomerativeClustering(n_clusters=3, linkage='ward')),
    ('Lunas',  X_m_sc, KMeans(n_clusters=2, n_init=10, random_state=42)),
    ('Lunas',  X_m_sc, DBSCAN(eps=0.3, min_samples=5)),
    ('Lunas',  X_m_sc, AgglomerativeClustering(n_clusters=2, linkage='ward')),
]

for ds_name, X_ds, model in configs:
    labels_r = model.fit_predict(X_ds)
    mask_r   = labels_r != -1
    n_cl     = len(set(labels_r[mask_r]))
    n_noise  = (labels_r == -1).sum()
    
    if n_cl > 1 and mask_r.sum() > 1:
        sil_r = silhouette_score(X_ds[mask_r], labels_r[mask_r])
    else:
        sil_r = float('nan')
    
    results.append({
        'Dataset': ds_name,
        'Algoritmo': type(model).__name__,
        'N° Clusters': n_cl,
        'N° Ruido': n_noise,
        'Silhouette': round(sil_r, 4) if not np.isnan(sil_r) else 'N/A'
    })

df_results = pd.DataFrame(results)
print('=== Tabla comparativa de algoritmos de clustering ===')
print(df_results.to_string(index=False))

---
## 8. Resumen y Conclusiones

En esta sesión profundizamos en los tres algoritmos principales de clustering:

### K-means
- Algoritmo iterativo que minimiza la inercia (WCSS).
- Rápido y escalable, pero asume clusters esféricos y convexos.
- Usar el **método del codo** y el **Silhouette score** para seleccionar $k$.
- Inicialización **K-means++** reduce la sensibilidad a la inicialización aleatoria.

### DBSCAN
- Basado en densidad: no requiere especificar $k$, maneja ruido y formas arbitrarias.
- Parámetros clave: `eps` (radio de vecindad) y `min_samples`.
- Seleccionar `eps` con la **gráfica k-distancia**.
- Menos eficiente computacionalmente ($O(n^2)$); no apto para datasets muy grandes sin índices espaciales.

### Clustering Jerárquico
- Construye una **jerarquía** de clusters visualizada en el **dendrograma**.
- El método de enlace **Ward** suele dar los mejores resultados para datos continuos.
- Permite elegir $k$ a posteriori cortando el dendrograma.
- Escalabilidad limitada ($O(n^2 \log n)$).

### Regla de oro
> No existe un algoritmo universalmente mejor. La elección depende de la **naturaleza de los datos**, el **tamaño del dataset**, la **forma esperada de los clusters** y los **requisitos del negocio**.

---

## Referencias

**Texto guía**
- Notas de clase del curso (disponibles en el repositorio).

**Bibliografía complementaria**
- Aggarwal, C. C., & Reddy, C. K. (Eds.). (2014). *Data Clustering: Algorithms and Applications* (Cap. 4–5). CRC Press.
- Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *KDD*, 96(34), 226–231.
- Arthur, D., & Vassilvitskii, S. (2007). k-means++: The advantages of careful seeding. *SODA 2007*, 1027–1035.
- Müllner, D. (2011). Modern hierarchical, agglomerative clustering algorithms. *arXiv preprint* arXiv:1109.2378.
- Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation of cluster analysis. *Journal of Computational and Applied Mathematics*, 20, 53–65.